# Differential Equations — Session 32
## Section 7.3: Operational Properties I

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to use the first translation theorem, complete squares for shifted transforms, define and graph unit-step functions, rewrite piecewise functions with steps, use the second translation theorem, solve delayed-forcing IVPs, and interpret delayed system responses.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–20 min | First translation theorem |
| 20–38 min | Shifted inverse transforms |
| 38–58 min | Unit-step functions |
| 58–78 min | Second translation theorem |
| 78–88 min | Delayed forcing in an IVP |
| 88–90 min | Exit check |

The beam example is an optional extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 7.3-A — First translation theorem

If

$$
\mathcal L\{f(t)\}=F(s),
$$

then

$$
\mathcal L\{e^{at}f(t)\}=F(s-a).
$$

Equivalently,

$$
\mathcal L^{-1}\{F(s-a)\}=e^{at}f(t).
$$

### Definition 7.3-B — Unit-step function

For $a>0$,

$$
u(t-a)=
\begin{cases}
0,&t<a,\\
1,&t\ge a.
\end{cases}
$$

### Theorem 7.3-C — Second translation theorem

If $\mathcal L\{f(t)\}=F(s)$, then

$$
\mathcal L\{u(t-a)f(t-a)\}
=
e^{-as}F(s).
$$

Equivalently,

$$
\mathcal L^{-1}\{e^{-as}F(s)\}
=
u(t-a)f(t-a).
$$

### Principle 7.3-D — Step representation of a piecewise function

If $f=g$ before $t=a$ and $f=h$ after $t=a$, then

$$
f(t)=g(t)+u(t-a)\big[h(t)-g(t)\big].
$$

### Classroom Checkpoint — Delayed Local Clock

Why does the second translation theorem use $f(t-a)$ in

$$
u(t-a)f(t-a)?
$$

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. First translation in the $s$-domain

Since

$$
\mathcal L\{\cos bt\}=\frac{s}{s^2+b^2},
$$

we have

$$
\mathcal L\{e^{at}\cos bt\}
=
\frac{s-a}{(s-a)^2+b^2}.
$$

In [ ]:
def exponential_shift(a=-1.0, b=3.0):
    t = np.linspace(0, 8, 700)
    f = np.exp(a*t)*np.cos(b*t)
    plt.plot(t, f)
    plt.xlabel("t")
    plt.ylabel("signal")
    plt.title(fr"$e^{{{a:.2f}t}}\cos({b:.2f}t)$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        exponential_shift,
        a=FloatSlider(min=-3, max=1, step=0.1, value=-1),
        b=FloatSlider(min=0.5, max=8, step=0.25, value=3)
    )
else:
    exponential_shift()

## 2. Complete the square before inverting

For example,

$$
\frac{3s+7}{s^2+4s+13}
=
\frac{3(s+2)+1}{(s+2)^2+9}.
$$

Therefore,

$$
f(t)=3e^{-2t}\cos3t+\frac13e^{-2t}\sin3t.
$$

In [ ]:
s, t = sp.symbols("s t", positive=True)
F = (3*s+7)/(s**2+4*s+13)
display(sp.inverse_laplace_transform(F, s, t))

## 3. Unit-step functions

In [ ]:
t_grid = np.linspace(0, 10, 800)
for a in [2, 5, 7]:
    plt.step(t_grid, unit_step(t_grid, a), where="post", label=fr"$u(t-{a})$")
plt.ylim(-0.1, 1.2)
plt.legend()
plt.title("Unit-step functions")
plt.show()

Suppose

$$
f(t)=
\begin{cases}
2t,&0\le t<3,\\
6,&t\ge3.
\end{cases}
$$

Then

$$
f(t)=2t+u(t-3)(6-2t).
$$

In [ ]:
t_grid = np.linspace(0, 8, 700)
piecewise = np.where(t_grid < 3, 2*t_grid, 6)
step_form = 2*t_grid + unit_step(t_grid, 3)*(6-2*t_grid)

plt.plot(t_grid, piecewise, label="piecewise definition")
plt.plot(t_grid, step_form, linestyle="--", label="unit-step form")
plt.legend()
plt.title("Equivalent representations")
plt.show()

print("maximum discrepancy:", np.max(np.abs(piecewise-step_form)))

## 4. Delayed signals

The function

$$
u(t-a)\sin\big(2(t-a)\big)
$$

is zero until $t=a$, then begins a sine wave with local clock $t-a$.

In [ ]:
def delayed_signal(a=3.0, omega=2.0):
    t = np.linspace(0, 12, 900)
    y = unit_step(t, a)*np.sin(omega*(t-a))
    plt.plot(t, y)
    plt.axvline(a, linestyle="--", label="activation time")
    plt.xlabel("t")
    plt.ylabel("signal")
    plt.title("Delayed oscillation")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        delayed_signal,
        a=FloatSlider(min=0, max=8, step=0.25, value=3),
        omega=FloatSlider(min=0.5, max=6, step=0.25, value=2)
    )
else:
    delayed_signal()

## 5. Delayed forcing in an IVP

Consider

$$
y''+4y=u(t-3),
\qquad
y(0)=0,
\qquad
y'(0)=0.
$$

The transform is

$$
Y(s)=\frac{e^{-3s}}{s(s^2+4)}.
$$

First invert the unshifted factor, then delay it.

In [ ]:
base_F = 1/(s*(s**2+4))
base_f = sp.simplify(sp.inverse_laplace_transform(base_F, s, t))
display(base_f)

In [ ]:
t_grid = np.linspace(0, 15, 900)
a = 3
base = 0.25*(1-np.cos(2*(t_grid-a)))
response = unit_step(t_grid, a)*base

plt.plot(t_grid, response)
plt.axvline(a, linestyle="--")
plt.xlabel("t")
plt.ylabel("y(t)")
plt.title("Response to a delayed unit step")
plt.show()

### Interactive delayed pulse

A rectangular pulse can be written as

$$
u(t-a)-u(t-b).
$$

In [ ]:
def pulse_response(a=2.0, b=5.0):
    if b <= a:
        print("Choose b>a.")
        return
    t = np.linspace(0, 15, 900)
    forcing = unit_step(t, a)-unit_step(t, b)

    def rhs(t, z):
        force = float(t >= a)-float(t >= b)
        return [z[1], force-4*z[0]]

    sol = solve_ivp(rhs, (0, 15), [0, 0], t_eval=t, rtol=1e-9, atol=1e-11)

    plt.plot(t, forcing, label="forcing")
    plt.plot(t, sol.y[0], label="response")
    plt.axvline(a, linestyle="--")
    plt.axvline(b, linestyle="--")
    plt.legend()
    plt.title("Finite-duration forcing and response")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        pulse_response,
        a=FloatSlider(min=0, max=8, step=0.25, value=2),
        b=FloatSlider(min=1, max=12, step=0.25, value=5)
    )
else:
    pulse_response()

## Optional extension — Piecewise beam load

A concentrated change in distributed load can be encoded with step functions and integrated repeatedly after transforming $EIy^{(4)}=w(x)$. Boundary conditions at the far endpoint determine the initially unknown bending moment and shear.

In [ ]:
# A simple illustrative clamped-beam deflection under a load that begins at x=a.
L, a = 10.0, 4.0
x = np.linspace(0, L, 700)
load = unit_step(x, a)

# Solve y'''' = load numerically as a first-order system with provisional left data.
def rhs(x, z):
    return [z[1], z[2], z[3], float(x >= a)]

# Two basis shots for unknown y''(0), y'''(0)
sol0 = solve_ivp(rhs, (0, L), [0, 0, 0, 0], t_eval=x)
solA = solve_ivp(rhs, (0, L), [0, 0, 1, 0], t_eval=x)
solB = solve_ivp(rhs, (0, L), [0, 0, 0, 1], t_eval=x)

M = np.array([[solA.y[0,-1]-sol0.y[0,-1], solB.y[0,-1]-sol0.y[0,-1]],
              [solA.y[1,-1]-sol0.y[1,-1], solB.y[1,-1]-sol0.y[1,-1]]])
rhs_bc = -np.array([sol0.y[0,-1], sol0.y[1,-1]])
c2, c3 = np.linalg.solve(M, rhs_bc)
deflection = sol0.y[0] + c2*(solA.y[0]-sol0.y[0]) + c3*(solB.y[0]-sol0.y[0])

plt.plot(x, deflection)
plt.axvline(a, linestyle="--", label="load begins")
plt.xlabel("x")
plt.ylabel("scaled deflection")
plt.title("Clamped beam with a step load")
plt.legend()
plt.show()

## Classroom Checkpoint — Exit Check

Find

$$
\mathcal L^{-1}\left\{
e^{-4s}\frac{2}{s^2+4}
\right\}.
$$

> Pause here. Let students commit to an answer before running the next cell.